# 04b: XGBoost

**Author:** Karan Homayounfar (25065219), UWE Bristol

XGBoost (Chen and Guestrin, 2016) represents the state of the art for tabular classification.  
It extends gradient boosting with second-order Taylor approximations, regularisation, and column subsampling, giving it a strong edge over standard Random Forest on many real-world datasets.  
Including XGBoost in the comparison completes the algorithm spectrum: linear (LR), bagging ensemble (RF), boosting ensemble (XGBoost), and kernel-based (SVM).

In [ ]:
import numpy as np
import pickle
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import xgboost as xgb
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    f1_score, roc_auc_score, classification_report,
    ConfusionMatrixDisplay, confusion_matrix,
    roc_curve, average_precision_score, precision_recall_curve
)

plt.rcParams.update({'figure.dpi': 100, 'axes.spines.top': False, 'axes.spines.right': False})

DATA_DIR    = '../data/processed'
FIGURES_DIR = '../report/figures'
import os
os.makedirs(FIGURES_DIR, exist_ok=True)

In [ ]:
X_train = np.load(f'{DATA_DIR}/X_train.npy')
X_test  = np.load(f'{DATA_DIR}/X_test.npy')
y_train = np.load(f'{DATA_DIR}/y_train.npy')
y_test  = np.load(f'{DATA_DIR}/y_test.npy')

with open(f'{DATA_DIR}/feature_names.pkl', 'rb') as f:
    feature_names = pickle.load(f)

# XGBoost handles class imbalance via scale_pos_weight
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight = neg_count / pos_count
print(f'scale_pos_weight = {scale_pos_weight:.2f}  (neg/pos ratio)')

## 1. Nested cross-validation

In [ ]:
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

param_grid = {
    'max_depth':    [4, 6],
    'learning_rate': [0.05, 0.1],
    'subsample':    [0.8],
    'colsample_bytree': [0.8],
}

base_xgb = xgb.XGBClassifier(
    n_estimators=300,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    use_label_encoder=False,
    n_jobs=-1,
    random_state=42,
    verbosity=0
)

xgb_inner = GridSearchCV(
    base_xgb,
    param_grid,
    cv=inner_cv,
    scoring='f1_macro',
    n_jobs=-1
)

outer_f1  = []
outer_auc = []
best_params_list = []

print('Running nested CV for XGBoost (5 outer x 3 inner)...')
for fold, (train_idx, val_idx) in enumerate(outer_cv.split(X_train, y_train), 1):
    X_tr, X_val = X_train[train_idx], X_train[val_idx]
    y_tr, y_val = y_train[train_idx], y_train[val_idx]

    xgb_inner.fit(X_tr, y_tr)
    best_params_list.append(xgb_inner.best_params_)

    y_pred = xgb_inner.predict(X_val)
    y_prob = xgb_inner.predict_proba(X_val)[:, 1]

    f1  = f1_score(y_val, y_pred, average='macro')
    auc = roc_auc_score(y_val, y_prob)
    outer_f1.append(f1)
    outer_auc.append(auc)
    print(f'  Fold {fold}: F1-macro={f1:.4f}, ROC-AUC={auc:.4f} | {xgb_inner.best_params_}')

print(f'\nNested CV F1-macro:  {np.mean(outer_f1):.4f} +/- {np.std(outer_f1):.4f}')
print(f'Nested CV ROC-AUC:  {np.mean(outer_auc):.4f} +/- {np.std(outer_auc):.4f}')

## 2. Final model on full training data

In [ ]:
from collections import Counter

def most_common_val(lst, key):
    return Counter([p[key] for p in lst]).most_common(1)[0][0]

best_max_depth    = most_common_val(best_params_list, 'max_depth')
best_lr           = most_common_val(best_params_list, 'learning_rate')

print(f'Final: max_depth={best_max_depth}, learning_rate={best_lr}')

xgb_final = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=best_max_depth,
    learning_rate=best_lr,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    use_label_encoder=False,
    n_jobs=-1,
    random_state=42,
    verbosity=0
)
xgb_final.fit(X_train, y_train)

y_pred_test = xgb_final.predict(X_test)
y_prob_test = xgb_final.predict_proba(X_test)[:, 1]

xgb_test_f1  = f1_score(y_test, y_pred_test, average='macro')
xgb_test_auc = roc_auc_score(y_test, y_prob_test)
xgb_test_pr  = average_precision_score(y_test, y_prob_test)

print(f'Test F1-macro:  {xgb_test_f1:.4f}')
print(f'Test ROC-AUC:   {xgb_test_auc:.4f}')
print(f'Test PR-AUC:    {xgb_test_pr:.4f}')
print()
print(classification_report(y_test, y_pred_test, target_names=['Low potential', 'High potential']))

## 3. XGBoost built-in feature importances

In [ ]:
importances = xgb_final.feature_importances_
top_n = 20
sorted_idx = np.argsort(importances)[::-1][:top_n]
top_feat_names = [feature_names[i] for i in sorted_idx]
top_vals       = importances[sorted_idx]

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(top_feat_names[::-1], top_vals[::-1], color='coral')
ax.set_xlabel('Feature importance (gain)')
ax.set_title('XGBoost: top 20 feature importances')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/xgb_feature_importances.png', bbox_inches='tight')
plt.show()

## 4. Save predictions and metrics

In [ ]:
np.save(f'{DATA_DIR}/xgb_test_probs.npy', y_prob_test)
np.save(f'{DATA_DIR}/xgb_test_preds.npy', y_pred_test)

xgb_final.save_model(f'{DATA_DIR}/xgb_final_model.json')

xgb_metrics = {
    'model': 'XGBoost',
    'best_max_depth': best_max_depth,
    'best_learning_rate': best_lr,
    'cv_f1_macro_mean': np.mean(outer_f1),
    'cv_f1_macro_std':  np.std(outer_f1),
    'cv_roc_auc_mean':  np.mean(outer_auc),
    'cv_roc_auc_std':   np.std(outer_auc),
    'test_f1_macro': xgb_test_f1,
    'test_roc_auc':  xgb_test_auc,
    'test_pr_auc':   xgb_test_pr,
}

with open(f'{DATA_DIR}/xgb_metrics.pkl', 'wb') as f:
    pickle.dump(xgb_metrics, f)

print('Saved XGBoost outputs.')
print('\n=== XGBoost SUMMARY ===')
for k, v in xgb_metrics.items():
    print(f'  {k}: {v}')